In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm

In [3]:
variable_names_path = "/content/drive/MyDrive/RTRP/Datasets/VariableNames.txt"
ransomware_data_path = "/content/drive/MyDrive/RTRP/Datasets/RansomwareData.csv"

In [4]:
raw_data_map = {}
with open(variable_names_path, "r", encoding="latin-1") as f:
    for line in f:
        parts = line.strip().split(";", 1)
        key = int(parts[0])
        value = parts[1]
        raw_data_map[key] = value

In [5]:
def reformat_registry_text(text):
    if text.startswith("REG:OPENED:"):
        return "opened registry " + text.split(":",2)[2]
    elif text.startswith("REG:READ:"):
        return "read registry " + text.split(":",2)[2]
    elif text.startswith("REG:WRITTEN:"):
        return "wrote registry " + text.split(":",2)[2]
    elif text.startswith("REG:DELETED:"):
        return "deleted registry " + text.split(":",2)[2]
    return text


def reformat_drop_text(text):
    if text.startswith("DROP:"):
        return "dropped file extension " + text.split(":")[1]
    return text


def reformat_api_text(text):
    text = text[4:]
    text = text.rstrip("AW")
    return "API " + text

In [6]:
for key in raw_data_map:

    if 4 <= key < 236:
        raw_data_map[key] = reformat_api_text(raw_data_map[key])

    elif 236 <= key < 582:
        raw_data_map[key] = reformat_drop_text(raw_data_map[key])

    elif 582 <= key < 7204:
        raw_data_map[key] = reformat_registry_text(raw_data_map[key])

In [7]:
df = pd.read_csv(ransomware_data_path, header=None)
print(df.shape)

(1524, 30970)


In [8]:
apiFeatures = []
dropFeatures = []
regFeatures = []
filesFeatures = []
filesEXTFeatures = []
dirFeatures = []
strFeatures = []

for _, row in tqdm(df.iterrows(), total=len(df)):

    parts = ["","","","","","",""]

    for col, val in enumerate(row.values[3:], start=3):

        if val == 1 and col+1 in raw_data_map:

            if col < 235:
                parts[0] += raw_data_map[col+1] + ". "

            elif col < 581:
                parts[1] += raw_data_map[col+1] + ". "

            elif col < 7203:
                parts[2] += raw_data_map[col+1] + ". "

            elif col < 11344:
                parts[3] += raw_data_map[col+1] + ". "

            elif col < 12279:
                parts[4] += raw_data_map[col+1] + ". "

            elif col < 14703:
                parts[5] += raw_data_map[col+1] + ". "

            else:
                parts[6] += raw_data_map[col+1] + ". "

    apiFeatures.append(parts[0])
    dropFeatures.append(parts[1])
    regFeatures.append(parts[2])
    filesFeatures.append(parts[3])
    filesEXTFeatures.append(parts[4])
    dirFeatures.append(parts[5])
    strFeatures.append(parts[6])

100%|██████████| 1524/1524 [00:10<00:00, 151.05it/s]


In [9]:
formatted_df = df[[2]].copy()

formatted_df["apiFeatures"] = apiFeatures
formatted_df["dropFeatures"] = dropFeatures
formatted_df["regFeatures"] = regFeatures
formatted_df["filesFeatures"] = filesFeatures
formatted_df["filesEXTFeatures"] = filesEXTFeatures
formatted_df["dirFeatures"] = dirFeatures
formatted_df["strFeatures"] = strFeatures

formatted_df.rename(columns={2: "family"}, inplace=True)

In [10]:
output_path = "/content/drive/MyDrive/RTRP/Datasets/after_feature_internal_semantic_process_data.csv"
formatted_df.to_csv(output_path, index=False)
print("Saved:", output_path)

Saved: /content/drive/MyDrive/RTRP/Datasets/after_feature_internal_semantic_process_data.csv


In [11]:
formatted_df.shape

(1524, 8)